# Audio Machine Learning - Workshop Week 7 - Dataset Processing

## Introduction

Now the summative coding assignment is finished, we will now focus more on topics that will be useful for your group projects.

In this session, you'll create custom audio data processing functions that you can use for the group project. The following workshop provides some structured approach to implementing a dataset preprocessing and feature extraction pipeline.

By the end of this session, you'll have a reusable pipeline that you can adapt and extend for your project work.

## 0 - Import Libraries

In [ ]:
%pip install librosa
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import IPython.display
from tqdm import tqdm
import glob
import random
import tqdm
import urllib.request
import zipfile

## Step 1 - Downloading the Data

For this workshop, I'll give some examples using the ESC-50 Environmental Sound Classification dataset

 https://github.com/karolpiczak/ESC-50

You may also use your own dataset if you already have one in mind for your project.

### Exercise 1.1: Dataset Download

You can implement a function to download your chosen dataset. Below is a template for ESC-50 that you can adapt for your chosen dataset. You are also welcome to just download the dataset manually if you like.

In [ ]:
def download_esc50():
    """Download ESC-50 dataset"""
    
    # Create a directory for datasets
    os.makedirs('datasets', exist_ok=True)
    
    # URL for the ESC-50 dataset
    url = 'https://github.com/karoldvl/ESC-50/archive/master.zip'
    # Where the zip file will be saved
    zip_path = 'datasets/ESC-50-master.zip'
    
    # Download the dataset if it doesn't exist
    if not os.path.exists(zip_path):
        print(f"Downloading ESC-50 dataset from {url}...")
        urllib.request.urlretrieve(url, zip_path)
        print("Download complete!")
        
        # Extract the zip file
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('datasets')
        print("Extraction complete!")
    else:
        print("ESC-50 dataset already downloaded.")
    
    return 'datasets/ESC-50-master'

In [ ]:
# TODO: Implement a download function for your chosen dataset
# Adapt the above or implement your own function to download your dataset

def download_dataset():
    """Download your chosen dataset"""
    # Your implementation here
    pass

In [ ]:
# Call your download function
dataset_path = download_dataset()  # Or download_esc50() if using ESC-50

### Exercise 1.2: Dataset Exploration

Now that you have your dataset, let's explore it to understand its characteristics. The code below provides a starting point, but you should adapt it to your specific dataset structure.

You can usually find this information out from a 'metadata' file, or something similar, that is often included with dataset.

In [ ]:
def explore_dataset(dataset_path):
    # TODO: Implement dataset exploration
    # This will vary based on your dataset structure, and whether the dataset includes a metadata file
    
    # Example for ESC-50:
    if 'ESC-50' in dataset_path:
        # Load metadata
        metadata = pd.read_csv(os.path.join(dataset_path, 'meta/esc50.csv'))
        
        # Print dataset statistics
        print(f"Total number of files: {len(metadata)}")
        print(f"Number of categories: {metadata['category'].nunique()}")
        print(f"Categories: {', '.join(sorted(metadata['category'].unique()))}")
        print(f"\nCategory distribution:")
        print(metadata['category'].value_counts())
        
        # Return paths to a few example files
        audio_dir = os.path.join(dataset_path, 'audio')
        example_files = [os.path.join(audio_dir, filename) for filename in metadata['filename'].sample(5)]
        return example_files
    else:
        # TODO: Adapt for your dataset
        print("Dataset exploration not implemented for this dataset.")
        return []

In [ ]:
example_files = explore_dataset(dataset_path)

## Step 2 - Audio Loading & Inspection

Now that we have our dataset, let's implement functions to load and inspect audio files. This will help us understand the data better and inform our processing decisions.

You should make a function that takes a filepath as argument. The function should load the chosen file using librosa, and optionally display the waveform/spectrogram. It should also return or print information such as the file length and sample rate, and whether it is stereo or mono.

In [ ]:
def load_and_inspect_audio(file_path, display=True):
    """Load an audio file and return basic information about it."""
    # Load the audio file
    y, fs = librosa.load(file_path, sr=None)  # Use the file's original sample rate
    
    ## TODO Get some basic information about the file that was just loaded
    
    if display:
        pass
        # Display a plot of the audio file, in the waveform and spectrogram domains
    
    return y, fs

In [ ]:
y, sr = load_and_inspect_audio(example_files[0], display=True)

Get a few more examples. Are the files all the same length?

## Step 3 - Audio Preprocessing

Before extracting features, it's often useful to preprocess the audio. This might include normalization, silence removal, or other transformations.

### Exercise 3.1: Preprocessing Function

Implement a preprocessing function that takes audio as input, and some arguments to determine the preprocessing applied. You should include options to, for example, trim trailing and leading silence (you can use librosa to do this), normalise the audio, adjust it to some target length (by truncating or zero-padding).

In [ ]:
def preprocess_audio(y, sr, target_length, normalise, trim_silence):
    """Apply preprocessing to audio data

    Args:
        y (ndarray): Audio time series
        sr (int): Sample rate
        target_length (float): Length of audio in seconds
        normalise (bool): Whether to normalise audio
        trim_silence (bool): Whether to trim leading/trailing silence
        
    Returns:
        ndarray: Preprocessed audio time series
    """
    # TODO: Implement preprocessing
    # Use librosa functions or otherwise to carry out preprocessing, based on the example arguments or any preprocessing needed for 
    # your dataset
    
    return y

### Exercise 3.2: Test your preprocessing function

In [ ]:
# Test your preprocessing function

# Load audio
y, sr = librosa.load(example_files[0])

# Apply preprocessing
y_preproc = preprocess_audio(y, sr)

# Use you plotting functions from before to plot the audio before and after preprocessing


In [ ]:
#Listen to the audio files before and after preprocessing
print("Original audio:")
IPython.display.Audio(data=y, rate=sr)

print("Preprocessed audio:")
IPython.display.Audio(data=y_preproc, rate=sr)

## Step 4 - Feature Extraction

Feature extraction is the step where you extract audio features for training your models. 


It can be convenient to store these arguments as a Python 'Dictionary', which can then be passed to the feature extraction function using the Python 'splat' operator:
Below is a template that can be adapted to call different feature extraction functions, with different arguments. This allows you to implement a single feature_extraction function that can select different features to extract depending on the arguments provided

In [ ]:
def a_function(argument_1, argument_2): # A function with some arguments
    print(argument_1)
    print(argument_2)
    
def another_function(argument_3, argument_4, argument_5): # A function with more arguments
    print(argument_3)
    print(argument_4)
    print(argument_5)
    
    
def feature_extractor(feature_type, arguments):
    
    if feature_type == 'a_function':
        print(f'calling {feature_type}')
        feats = a_function(**arguments) # ** is known as the 'splat' operator, the keys of the dictionary are provided as keywords and entries for the keys are provided as the arguments
    elif feature_type == 'another_function':
        print(f'calling {feature_type}')
        feats = another_function(**arguments)
        

#This allows you to use a single function to call various feature extractions, potentially with different numbers of arguments
feature_extractor('a_function', {'argument_1': 1.0, 'argument_2': 2.0})

feature_extractor('another_function', {'argument_3': 3.0, 'argument_4': 4.0, 'argument_5': 5.0})
        

    

### Exercise 4.1: Feature Extraction Functions

Implement feature extraction functions relevant to your chosen task. The function should take the audio, and sample rate as argument, as well as the chosen feature type and the arguments for that feature (i.e window size, hop length, etc). You can adapt the above structure if you like. Add whatever features you might like to try.

In [ ]:
def extract_features(y, sr, feature_type, feat_args):
    """Extract audio features
    
    Args:
        y (ndarray): Audio time series
        sr (int): Sample rate of audio
        feature_type (str): Type of feature to extract
        **kwargs: Additional arguments for feature extraction
        
    Returns:
        ndarray: Extracted features
    """
    # TODO: Implement feature extraction for your chosen features
    # Below is a couple of examples to get you started
    if feature_type == 'mfcc':
        pass
        # Implement mfcc extraction with arguments from feat_args
    elif feature_type == 'melspectrogram':
        pass
        # Implement melspectrogram extraction with arguments from feat_args
    else:
        raise ValueError(f"Unsupported feature type: {feature_type}")
        
    return # Return the features extracted from the audio

You can use this pattern to create a flexible configuration, for example, if you want to extract 'mfccs'

In [ ]:
feat_type = 'mfccs'
feat_args = {'sr':22050, 'n_mfcc': 20}

feature_extractor(feat_type, feat_args) #You will need to define the feature_extractor so it extracts 'feat_type' features using the arguments in 'feat_args'

In [ ]:
# Test feature extraction here


### Exercise 4.2: Feature Aggregation (Optional)

Depending on the task and model, you might need to aggregate features over time. Implement a function to compute statistics over the time dimension of features.


In [ ]:
def aggregate_features(features, aggregation_type):
    """Aggregate features over time dimension
    
    Args:
        features (ndarray): Feature matrix (n_features, n_frames)
        aggregation_type (str): Type of aggregation
        
    Returns:
        ndarray: Aggregated features vector
    """
    # TODO: Implement feature aggregation if needed for your task
    
    if aggregation_type == 'mean_std':
        # Compute mean and standard deviation along time axis
        return agregated_features
    
    elif aggregation_type == 'mean_min_max':
        # Compute mean, max and min of each feature along the time axis
        return agregated_features
    
    else:
        raise ValueError(f"Unsupported aggregation type: {aggregation_type}")
    
    return None

In [ ]:
# Test feature aggregation

# Extract features
features = extract_features(y_preproc, sr, feature_type='mfcc', feat_args={})
    
# Aggregate features
aggregation_types = ['mean_std', 'mean_min_max']
    
for agg_type in aggregation_types:
    pass
    # extract the features and aggregate them, then inspect the shapes of the outputs

## 5 - Building a Complete Processing Pipeline

Now we can combine all the components into a complete pipeline that can be applied to your dataset.

### Exercise 5.1: Audio Processing Pipeline

Implement a complete audio processing pipeline class that encapsulates all the steps.

In [ ]:
class AudioPipeline:
    """Audio processing pipeline"""
    
    def __init__(self, target_sr=22050, feature_type='mfcc', feat_args={}, aggregation_type=None):
        """Initialize the pipeline with parameters
        
        Args:
            target_sr (int): Target sample rate
            feature_type (str): Type of feature to extract
            feat_args (dict): Additional parameters for feature extraction
        """
        self.target_sr = target_sr
        self.feature_type = feature_type
        self.feat_args = feat_args
        self.aggregation_type = aggregation_type
    
    def process_file(self, file_path):
        """Process a single audio file through the pipeline
        
        Args:
            file_path (str): Path to the audio file
            normalize (bool): Whether to normalize features
            
        Returns:
            ndarray: Extracted features
        """
        # Use the previously defined functions to load an audio file, run preprocessing and extract the chosen features. Optionally apply feature aggregation.
    
    ## TODO add approriate arguments to the process_dataset method
    def process_dataset(self, file_list):
        """This method should take a list of files to be loaded.
        """
        features_list = []

        
        for file_path in tqdm(file_list, desc='Processing files'):
            pass
            #Do feature extraction here
        
        return features_list


### Exercise 5.2: Test the Pipeline

Test your pipeline on a few example files.

In [ ]:
# Test your pipeline here

## Step 6 - Applying to Your Project

Now that you have a complete audio processing pipeline, you can apply it to your project.

### Exercise 6.1: Process Your Dataset

Apply your pipeline to process your entire dataset (or a significant portion of it).

In [ ]:
# TODO: Modify this to work with your dataset
def get_all_audio_files(dataset_path):
    """Get all audio files in the dataset"""
    # This function will need to be customized for your dataset structure
    return all_files # List of all file paths in dataset

In [ ]:
# Get all audio files
all_files = get_all_audio_files(dataset_path)

# Limit to a subset for testing (optional)
max_files = 20  # Adjust based on your dataset size and available time
if len(all_files) > max_files:
    subset_files = random.sample(all_files, max_files)
    print(f"Using {len(subset_files)} files for testing")
else:
    subset_files = all_files

# Create an optimized pipeline for your task
# TODO: Customize these parameters for your specific task
pipeline = AudioPipeline(# Add arguments)



### Exercise 6.2: Save Processed Features (Optional)

Optionally, save the processed features to disk for future use.

In [ ]:
def save_features(features_list, file_list, output_dir):
    pass
    # Implement a function that will save all your extracted features to disk
    # You can either save them as one large numpy array, or as one numpy array per audio file in your dataset


In [ ]:
# Use the save features function here

## 7 - Next Steps

Now that you have a working audio processing pipeline, here are some suggestions for next steps:

1. **Dataset Organization**: Organize your dataset into train/validation/test splits
2. **Feature Selection**: Experiment with different feature types and parameters
3. **Model Selection**: Choose appropriate models for your task
4. **Transfer Learning**: Explore using pre-trained audio models (next workshop topic)
5. **Evaluation**: Plan how you'll evaluate your system's performance

### Exercise 7.1: Document Your Approach

Take some time to document your approach and decisions for your project.

### Exercise 7.2: Put useful functions in the 'data_functions.py' module

A useful way to organise code can be to put helpful functions that you've made in this worksheet in a Python module, allowing you to import them and use them in other scripts.

In [ ]:
import data_functions # e.g 'data_functions' can be populated with your functions/classes and imported like this
data_functions.a_useful_function()

from data_functions import a_useful_function # or imported like this!
a_useful_function()

##TODO Add function to the 'data_function.py' file included with this worksheet

## Conclusion

In this workshop, you've built a complete audio processing pipeline that's customised for your specific project needs. This is just intended to give you some ideas about how to approach working with your dataset. Please feel free to modify or use different approaches as is appropriate. 

